# 答辩演示流程（阶段 13：连续自适应 + HDMI 分屏）

## 演示前 5 分钟

- [ ] 显示器接好 PYNQ-Z2 的 HDMI，输入源切到对应的 HDMI 口
- [ ] 板子联网，`/home/xilinx/jupyter_notebooks/ai_video_2024.2/` 是完整目录
- [ ] **Jupyter 里只留这一个 notebook**（其它全部 shutdown，避免抢 VDMA/CPU）
- [ ] `Restart Kernel`（新内核，干净状态）
- [ ] 关掉可能残留的调度脚本：在终端 `pkill -f auto_schedule.py`
- [ ] （可选）台灯 + 纸盒暗室就位：第 2、3 幕的明暗差会更明显

## 演示脚本（按顺序做）

1. 跑「第 1 幕」：展示 overlay 与 IP 清单、VERSION（`0x0001020D` = 阶段 13）
2. 跑「第 2 幕」：展示 MLP 实时判定（遮镜头→0 / 逆光→1 / 移动→2）
3. 跑「幕 2.5」：HDMI 分屏 —— 左半原始、右半处理后，**同一区域**并排比
4. 跑「第 3 幕」：自动调度，画面自动切换（三种效果肉眼可见）
5. 跑「第 4 幕」：展示 `features.csv` 与训练准确率、量化一致率
6. 跑「收尾」：停掉一切，回到干净状态

## 第 1 幕：这台板子上跑的是什么

要点：`VERSION` 最低字节就是硬件阶段号，一眼证明板子上是最后那一版 bitstream。

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib import exposure
from lib.ai_video_regs import AiVideoRegs, STATUS_DONE

%matplotlib inline

# ---- 演示必须和采集用同一档曝光（单一来源 lib/exposure.py 的 WORKPOINT）----
# 不锁的话 AEC/AGC 会自动补偿：遮住镜头时它不是变暗，而是把画面顶回正常亮度。
# 实测 2026-10-15 演示日志：遮住画面后 f_luma 还有 41（暗光类的中心是 10），
# MLP 判成常规档、整场只在 1/2 之间跳 —— 「暗光增强」这一幕根本演不出来。
# 画面太暗/全黑时把开关改成 False（退回自动曝光：能看见画面，但特征会偏移）。
USE_MANUAL_EXPOSURE = True

ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(
    ROOT, manual_exposure=(dict(exposure.WORKPOINT) if USE_MANUAL_EXPOSURE else None))
vs.stop_all_channels(vdma0, vdma1)
vp.start_capture(vdma0, mode)

regs = AiVideoRegs(ol.axis_video_proc_0)
print("=" * 60)
print("overlay :", vs.find_overlay(ROOT))
print("VERSION : 0x%08X   -> 硬件阶段 %d" % (regs.version(), regs.version_stage()))
print("MODE_SEL:", regs.mode_sel())
print("IP 清单 :", ", ".join(sorted(ol.ip_dict.keys())))
print("=" * 60)

## 第 2 幕：MLP 实时判定

请评委/观众配合改变场景，观察 `MLP_OUTPUT` 与三个特征。
（label 含义：0=暗光，1=逆光，2=常规/细节）

In [ ]:
LABELS = {0: "暗光(增强模式0)", 1: "逆光(补偿模式1)", 2: "常规/细节(锐化模式2)"}

def show_mlp(seconds=25.0, poll=0.3):
    t0 = time.time()
    while time.time() - t0 < seconds:
        mlp = regs.mlp_output()
        f1, f2, f3 = regs.mlp_features()
        print("MLP=%d %-22s f=(%3d,%6d,%3d)" % (mlp, LABELS.get(mlp, "?"), f1, f2, f3))
        time.sleep(poll)

show_mlp(25.0)

In [ ]:
raw = vp.capture_frame(vdma0)
rgb = vp.frame_to_rgb(raw)
plt.figure(figsize=(10, 6))
plt.imshow(rgb)
plt.title("OV5640 live frame (post-PROC)")
plt.axis("off")
plt.show()
print("frame shape:", raw.shape, " mean:", round(float(raw.mean()), 1))

## 幕 2.5：HDMI 分屏对比（阶段 13）

**左半 = 原始画面，右半 = 处理后的同一区域**，1:1 不缩放，一眼就能比出差别。

* 开：`regs.set_split(True)`；关：`regs.set_split(False)`（关掉后和阶段 12 一模一样）；
* 下面两格**按现场挑一个演**：
  * **瞬时对比** `split_show(0, 255)` —— 右半增益 ×1.99，明暗差最直观；
  * **连续调光** `param_ramp()` —— 右半亮度平滑升降，证明参数是**连续**的（不是三档跳变）；
* 注意：分屏只显示固定的那 640 源列，**演示前把主体挪进这半幅画面**；
* 这一幕演完、下一幕（自动调度）之前，记得关掉分屏，否则整屏自动切换的效果看不全。

In [ ]:
def split_show(mode_sel=0, param=255, split=True):
    """现场切分屏：mode_sel 0=增益  1=逆光补偿  2=锐化。"""
    regs.set_split(split)
    regs.set_mode(mode_sel)
    regs.set_param(param)
    regs.wait_new_frame()
    print("split=%d  mode=%d  PARAM=%d" % (regs.split(), regs.mode_sel(), regs.param()))


vp.start_hdmi_passthrough(vdma0, vdma1, mode)   # 确保显示通路在跑（重复调用无害）
# 先判断硬件支不支持分屏：只有阶段 13 有（阶段 12 的 bit 把这一位当没接）。
# 不判断的话 set_split(True) 读回来是 1、屏幕却纹丝不动 —— 假成功最坑。
if not regs.has_split():
    print("!! 当前 bit 是阶段 %d，没有分屏硬件 -> 这一幕跳过。" % regs.version_stage())
    print("   要演分屏就重出阶段 13： bash build_overlay.sh 13（约 30~40 分钟）")
else:
    split_show(0, 255)      # 主画面：右半被提亮，左半保持原始
    # split_show(2, 255)    # 想展示锐化就换成这一行
# split_show(1, 255)    # 想展示逆光补偿就换成这一行

In [ ]:
import time


def param_ramp(lo=0, hi=255, step=8, dwell=0.06):
    """PARAM 从 lo 平滑推到 hi 再推回来：右半亮度连续变化，左半纹丝不动。"""
    regs.set_split(True)
    regs.set_mode(0)
    for v in list(range(lo, hi + 1, step)) + list(range(hi, lo - 1, -step)):
        regs.set_param(v)
        time.sleep(dwell)
    regs.set_param(hi)
    print("PARAM 连续扫描完成：%d -> %d -> %d" % (lo, hi, hi))


if regs.has_split():
    param_ramp()
else:
    print("（跳过 param_ramp：当前阶段 %d 没有分屏硬件）" % regs.version_stage())

# 演完这一幕，回到整屏（下一幕要看不分屏的自动切换）
regs.set_split(False)
print("split =", regs.split())

## 第 3 幕：自动调度（画面自动切换）

换场景时，系统会在**连续 3 次**判定一致后，在**帧边界**切换模式。

正式演示用独立进程跑（更稳）。注意：**不要**先释放 DMA 通道 —— 那会让 HDMI 黑屏，
而这一幕的看点正是“屏幕上的画面随场景自动切换”。下面用 `--base` 模式：
脚本只戳自有 IP 的寄存器，完全不碰 VDMA / I2C，采集+显示通路保持不动。

In [ ]:
import subprocess

# ---- 自动调度：和阶段 8 一样，用独立进程 + --base 模式 --------------------
# 三个坑都在阶段 8 踩过，演示时不能再犯：
#   1) 不要写死 python3 —— 板子上的 python3 可能是系统解释器（没有 pynq），
#      脚本会秒退，而 `> /dev/null` 把错误信息一起吞掉，看起来像“什么都没发生”；
#      要用 sys.executable，并把 stdout/stderr 写到文件里。
#   2) 不要先 stop 掉所有 VDMA 通道 —— 那样 HDMI 就黑了，而这一幕的看点
#      正是“屏幕上的画面随场景自动切换”。这里先 start_hdmi_passthrough()
#      开显示，脚本用 --base 只戳寄存器，完全不碰 VDMA / I2C。
#   3) 日志要留全：验收看的最后一行是收尾行，不能被截断。
LOG    = "/home/xilinx/sched.log"
STDOUT = "/home/xilinx/sched_stdout.log"
SCRIPT = os.path.join(ROOT, "stage08_auto_schedule", "auto_schedule.py")
BASE   = ol.ip_dict["axis_video_proc_0"]["phys_addr"]   # PYNQ 的键名是 phys_addr

# 先开显示通路：屏幕上要能看见画面（这一幕的“执行”就靠它）
regs.set_split(False)   # 关掉分屏：这一幕要让整屏自动切换，效果才看得全
vp.start_hdmi_passthrough(vdma0, vdma1, mode)

# 清掉上次的进程和日志，保证日志里只有本轮
subprocess.run(["bash", "-c", "pkill -f auto_schedule.py"], capture_output=True)
open(LOG, "w").close()
open(STDOUT, "w").close()

cmd = [sys.executable, SCRIPT,
       "--base", "0x%X" % BASE,       # 不加载 overlay / 不碰 VDMA / 不碰 I2C
       "--duration", "120",
       "--debounce", "8", "--min-hold", "2.0",
       "--heartbeat", "5",
       "--log", LOG]
print("启动:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=open(STDOUT, "a"), stderr=subprocess.STDOUT)
print("pid =", proc.pid)
print("现在开始换场景：遮镜头 → MLP=0；对着窗/灯 → 1；正常移动物体 → 2")
print("屏幕上的画面会自动切换，约 120 秒后自动结束（日志见下一格）")

In [ ]:
# 反复运行本单元刷新日志（演示时每隔十几秒点一次）
print(open(LOG).read())
print("---- 脚本的 stdout/stderr（有报错就看这里）----")
print(open(STDOUT).read()[-800:])
# 想提前结束：subprocess.run(["bash", "-c", "pkill -f auto_schedule.py"])

## 第 4 幕：数据与模型

展示 `features.csv` 的统计与训练/量化指标。

In [ ]:
import csv

csv_path = os.path.join(ROOT, "data", "features.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(ROOT, "data", "features_demo.csv")
print("数据文件:", csv_path)

rows = list(csv.reader(open(csv_path, encoding="utf-8")))
print("表头:", rows[0], "  样本数:", len(rows) - 1)

w = os.path.join(ROOT, "weights", "mlp_weights.h")
head = open(w, encoding="utf-8").read().splitlines()
print("权重文件:", w)
for line in head[:8]:
    print("   ", line)

print("\n训练/量化指标请回看 03_mlp_train.ipynb 与 04_mlp_quant_export.ipynb 的输出：")
print("   训练准确率 > 85%，量化前后一致率 > 97%，int32_ok = True")

## 收尾

演示结束一定要做，否则下一次演示会“莫名打不开 video”。

In [ ]:
subprocess.run(["bash", "-c", "pkill -f auto_schedule.py"], capture_output=True)
print("调度进程已停止")
# 如果还要继续用 video，重新加载一次 overlay；否则 Restart Kernel 即可。
print("提示：下一步若要继续操作，请 Restart Kernel 后重新加载 overlay")